# Video 05 - Answers code can use: structured JSON output

**You are here:** Video 5 of 13 - the last of the Foundations (see `../you-are-here.svg`).

**The one idea:** people read text. **Code reads JSON.** A JSON schema makes the model answer in a shape your code can rely on - but your code still checks the values.

**By the end you can:**
1. Explain why agents need JSON, not free text.
2. Turn a JSON reply into a Python `dict`.
3. Use a JSON schema (`response_format`) to control the shape of the answer.
4. Extract data from messy text - and check the values before trusting them.

**Before you start:** attach this notebook to **Serverless**.

In [0]:
import json
from databricks.sdk import WorkspaceClient

client = WorkspaceClient().serving_endpoints.get_open_ai_client()
MODEL = "databricks-meta-llama-3-3-70b-instruct"
OTHER_MODEL = "databricks-claude-haiku-4-5"
print("Connected. Using models:", MODEL, "and", OTHER_MODEL)

REVIEW = "The battery died after one day and support never replied."

## Part 1 - Free text is for people
An agent has to *decide* things from the model's answer. Let's see what a normal answer looks like (diagram: `../diagrams/01-text-vs-json.svg`).

In [0]:
response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": f"What is the sentiment of this review and how confident are you? {REVIEW}"}],
)
print(response.choices[0].message.content)
# Easy for you to read. But how would an `if` statement find the label and the number?

## Way 1 - Ask nicely for JSON
We describe the JSON we want in the prompt, then parse each reply with `json.loads` (diagram: `../diagrams/02-three-ways-to-get-json.svg`).

**Predict first:** will both models' replies parse?

In [0]:
prompt = (
    f'Classify the sentiment of this review: "{REVIEW}"\n'
    'Reply ONLY with JSON like {"sentiment": "positive|negative|neutral", "confidence": 0.0-1.0}'
)

replies = {}
for model in [MODEL, OTHER_MODEL]:
    text = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}], temperature=0
    ).choices[0].message.content
    replies[model] = text
    try:
        print(f"{model}: parsed OK -> {json.loads(text)}")
    except json.JSONDecodeError:
        print(f"{model}: FAILED to parse. Raw reply: {text!r}")

## Way 2 - Clean up the reply
Some models wrap JSON in a markdown **code fence** (three backticks + `json`). We can strip it before parsing. Better - but we're guessing at every way a model might decorate its answer.

In [0]:
def parse_json(text):
    """Parse JSON, first removing a ```json ... ``` code fence if there is one."""
    text = text.strip()
    if text.startswith("```"):
        text = text.strip("`").removeprefix("json").strip()
    return json.loads(text)

for model, text in replies.items():
    print(f"{model}: {parse_json(text)}")

## Way 3 - A JSON schema (recommended)
Describe the exact shape - fields, types, allowed values - and pass it as `response_format`. The model is **constrained** to that shape.

| Schema piece | Meaning |
| --- | --- |
| `"enum": [...]` | must be one of these values |
| `"type": "number"` | must be a number |
| `"required": [...]` | these fields must be present |
| `"additionalProperties": False` | no extra fields |

In [0]:
sentiment_format = {
    "type": "json_schema",
    "json_schema": {
        "name": "sentiment",
        "strict": True,
        "schema": {
            "type": "object",
            "properties": {
                "sentiment": {"type": "string", "enum": ["positive", "negative", "neutral"]},
                "confidence": {"type": "number"},
            },
            "required": ["sentiment", "confidence"],
            "additionalProperties": False,
        },
    },
}

def classify(review, model=MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": f"Classify the sentiment of this review: {review}"}],
        response_format=sentiment_format,
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)   # plain json.loads - no clean-up needed

for model in [MODEL, OTHER_MODEL]:
    result = classify(REVIEW, model)
    print(f"{model}: {result}  (type: {type(result).__name__})")

In [0]:
# Now the model's answer can drive normal Python logic
for review in ["I love it, works perfectly!", "Terrible support, I want a refund.", "It arrived on Tuesday."]:
    result = classify(review)
    if result["sentiment"] == "negative" and result["confidence"] > 0.7:
        action = "ESCALATE to a human"
    else:
        action = "no action"
    print(f"{review:38} -> {result['sentiment']:8} {result['confidence']:.2f} -> {action}")

## Part 2 - Extracting data from messy text
The same trick pulls fields out of messages people write however they like. Fields that may be missing are allowed to be `null`: `{"type": ["string", "null"]}`.

In [0]:
import pandas as pd

def booking_format(allow_null=True):
    kind = (lambda t: {"type": [t, "null"]}) if allow_null else (lambda t: {"type": t})
    return {
        "type": "json_schema",
        "json_schema": {
            "name": "booking",
            "strict": True,
            "schema": {
                "type": "object",
                "properties": {"name": kind("string"), "people": kind("integer"), "date": kind("string"), "time": kind("string")},
                "required": ["name", "people", "date", "time"],
                "additionalProperties": False,
            },
        },
    }

EXTRACT = "Extract the booking details from the message. If a detail is not in the message, use null. Never guess."

def extract_booking(message, model=MODEL, allow_null=True):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "system", "content": EXTRACT}, {"role": "user", "content": message}],
        response_format=booking_format(allow_null),
        temperature=0,
    )
    return json.loads(response.choices[0].message.content)

MESSAGES = [
    "Table for 2 please, Saturday 7:30pm, name's Tom",
    "hey its Mei - can we get a booking for six people on the 14th at 12 noon?",
    "Hi, this is Priya. Can I book a table for 4 people this Friday?",   # no time!
]
display(pd.DataFrame([extract_booking(m) for m in MESSAGES], index=[m[:35] for m in MESSAGES]))

## Cost meter: JSON is short
Output tokens cost money. A JSON answer has no greeting, no explanation - just the data.

**Predict first:** how many times fewer output tokens will the JSON answer use?

In [0]:
question = [{"role": "user", "content": f"What is the sentiment of this review and how confident are you? {REVIEW}"}]

prose = client.chat.completions.create(model=MODEL, messages=question)
as_json = client.chat.completions.create(model=MODEL, messages=question, response_format=sentiment_format)

print(f"Prose answer : {prose.usage.completion_tokens:3} output tokens")
print(f"JSON answer  : {as_json.usage.completion_tokens:3} output tokens -> {as_json.choices[0].message.content}")
print(f"JSON used {prose.usage.completion_tokens / as_json.usage.completion_tokens:.1f}x fewer output tokens")

## What could go wrong? Valid JSON is not correct JSON
Priya's message has **no time**. What does each model put in `time`? Let's try with `time` forced to be a string, and with `null` allowed (diagram: `../diagrams/03-valid-is-not-correct.svg`).

**Predict first:** will you get Python's `None` every time?

In [0]:
priya = MESSAGES[2]

for model in [MODEL, OTHER_MODEL]:
    for allow_null in [False, True]:
        booking = extract_booking(priya, model=model, allow_null=allow_null)
        print(f"{model:40} null allowed={allow_null!s:5} -> time={booking['time']!r:12} "
              f"| if booking['time'] is {bool(booking['time'])}")

# A non-empty string like "null" or "<UNKNOWN>" is truthy - your code would think it HAS a time.

In [0]:
PLACEHOLDERS = {"", "null", "none", "<unknown>", "unknown", "n/a"}

def clean(data):
    """Turn placeholder strings into a real None so missing values are really missing."""
    return {k: None if isinstance(v, str) and v.strip().lower() in PLACEHOLDERS else v for k, v in data.items()}

for model in [MODEL, OTHER_MODEL]:
    booking = clean(extract_booking(priya, model=model, allow_null=False))
    missing = [field for field, value in booking.items() if value is None]
    print(f"{model:40} -> {booking}")
    print(f"{'':40}    missing: {missing} -> ask the user for: {', '.join(missing) or 'nothing'}")

## Recap
* Agents are code - they need **JSON**, not prose. `json.loads` turns it into a `dict`.
* **Asking nicely** is fragile (code fences, extra words). **A JSON schema** (`response_format`) is reliable.
* JSON answers are **short** - far fewer output tokens.
* A schema guarantees the **shape**, not the **truth**: check values in code (`clean()`).

## The bridge to tools
When a model "calls a tool", it does exactly what you saw today: it returns structured output. The structure is just **which function** and **which arguments**.

## Practise
* [exercises.md](../exercises.md) - 3 exercises + bonus (about 15 minutes). Add your code below this cell.
* [quiz.md](../quiz.md) - 5 questions.
* Solutions: `05-json-answers-solutions` (try first!).

**Next:** Video 06 - Giving the model hands: your first tool.